# Build Speaker-Level Sign Count Dataset

This notebook aggregates the utterance-level `sign_count` dataset into one row per speaker, system, and target class by averaging the concept columns across that speaker's utterances.


In [ ]:
from pathlib import Path

import pandas as pd


In [ ]:
# ===== Config =====
PROJECT_ROOT = Path('/home/SpeakerRec/BioVoice')
TCAV_ROOT = PROJECT_ROOT / 'redimnet' / 'tcav' / 'captum_tcav' / 'asvspoof5'
OUTPUT_SUBDIR = 'subset_20spk_20utts_per_spk_one_logistic_head'
OUTPUT_DIR = TCAV_ROOT / 'output' / OUTPUT_SUBDIR

INPUT_CSV = OUTPUT_DIR / 'merged_tcav_sign_count_wide.csv'
OUTPUT_CSV = OUTPUT_DIR / 'merged_tcav_sign_count_speaker_level.csv'

# Set True if you want to exclude A12 here.
EXCLUDE_A12 = False

print('INPUT_CSV =', INPUT_CSV)
print('Exists =', INPUT_CSV.exists())
print('OUTPUT_CSV =', OUTPUT_CSV)


In [ ]:
df = pd.read_csv(INPUT_CSV)

if EXCLUDE_A12:
    df = df[df['system_id'] != 'A12'].copy()

meta_cols = ['utt_id', 'speaker_id', 'split', 'source_partition', 'system_id', 'target_class', 'binary_label']
feature_cols = [c for c in df.columns if c not in meta_cols]

print('Input rows =', len(df))
print('Input utterances =', df['utt_id'].nunique())
print('Input speakers =', df['speaker_id'].nunique())
print('Feature columns =', len(feature_cols))
display(df.head())


In [ ]:
group_cols = ['speaker_id', 'split', 'source_partition', 'system_id', 'target_class', 'binary_label']

speaker_df = (
    df.groupby(group_cols, as_index=False)[feature_cols]
    .mean()
)

utt_counts = (
    df.groupby(group_cols, as_index=False)
    .agg(n_utterances=('utt_id', 'nunique'))
)

speaker_df = speaker_df.merge(
    utt_counts,
    on=group_cols,
    how='left',
    validate='one_to_one',
)

speaker_df = speaker_df.sort_values(['source_partition', 'system_id', 'target_class', 'speaker_id']).reset_index(drop=True)

print('Speaker-level rows =', len(speaker_df))
print('Speaker-system-class groups =', len(speaker_df))
print('Mean utterances per group =', speaker_df['n_utterances'].mean())
display(speaker_df.head())


In [ ]:
summary_df = pd.DataFrame({
    'rows': [len(speaker_df)],
    'speakers': [speaker_df['speaker_id'].nunique()],
    'systems': [speaker_df['system_id'].nunique()],
    'classes': [speaker_df['target_class'].nunique()],
    'feature_cols': [len(feature_cols)],
    'mean_n_utterances': [speaker_df['n_utterances'].mean()],
    'min_n_utterances': [speaker_df['n_utterances'].min()],
    'max_n_utterances': [speaker_df['n_utterances'].max()],
})
display(summary_df)

display(
    speaker_df.groupby(['system_id', 'target_class'])['speaker_id']
    .nunique()
    .rename('n_speakers')
    .reset_index()
    .sort_values(['system_id', 'target_class'])
)


In [ ]:
speaker_df.to_csv(OUTPUT_CSV, index=False)
print('Saved speaker-level sign-count dataset to:', OUTPUT_CSV)
